# Library Examples

## Archiving information

In [ ]:
from datetime import datetime, timedelta

import polars as pl
from pytz import UTC

from epicsarchiver import ArchiverAppliance

%matplotlib inline

In [ ]:
import json
import threading
import urllib.parse
from http.server import BaseHTTPRequestHandler, HTTPServer

from _fake_data import create_pb_bytes

_SEARCH_RESULTS = ["EXAMPLE:TEMPERATURE", "EXAMPLE:TEMPERATURE2", "EXAMPLE:PRESSURE"]


class _MockHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        qs = urllib.parse.parse_qs(parsed.query)
        if "getMatchingPVs" in parsed.path:
            body = json.dumps(_SEARCH_RESULTS).encode()
            content_type = "application/json"
        else:
            pv = qs.get("pv", ["unknown"])[0]
            body = create_pb_bytes(pv)
            content_type = "application/octet-stream"
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass


class _ReusableHTTPServer(HTTPServer):
    allow_reuse_address = True


_mock_server = _ReusableHTTPServer(("localhost", 17668), _MockHandler)
threading.Thread(target=_mock_server.serve_forever, daemon=True).start()


In [ ]:
archiver = ArchiverAppliance("localhost")
pv = "EXAMPLE:TEMPERATURE"

## Getting Data

In [ ]:
_, events = archiver.get_events(pv, datetime.now(tz=UTC) - timedelta(seconds=1), datetime.now(tz=UTC))
events

In [ ]:
df = archiver.get_data(pv, datetime.now(tz=UTC) - timedelta(seconds=30), datetime.now(tz=UTC))

In [ ]:
df.head()

## Async Fetch Data

In [ ]:
from epicsarchiver.retrieval.client.async_archiver_retrieval import AsyncArchiverRetrieval
from pytz import timezone

In [ ]:
tz = timezone("Europe/Stockholm")

In [ ]:
async with AsyncArchiverRetrieval(archiver.hostname) as a_archiver:
    print(await a_archiver.get_events(pv, datetime.now(tz=tz) - timedelta(microseconds=100), datetime.now(tz=tz)))

In [ ]:
async with AsyncArchiverRetrieval(archiver.hostname) as a_archiver:
    print(await a_archiver.get_all_events([pv, "EXAMPLE:TEMPERATURE2", "EXAMPLE:TEMPERATURE3"], datetime.now(tz=tz) - timedelta(microseconds=100), datetime.now(tz=tz)))

## Displaying and Calculating Summaries

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df["date"], df["val"])
plt.xlabel("time")
plt.ylabel("val")
plt.tight_layout()
# NBVAL_IGNORE_OUTPUT

In [ ]:
from epicsarchiver.retrieval.client.processor import Processor, ProcessorName

In [ ]:
df_mean = archiver.get_data(
    pv,
    datetime.now(tz=UTC) - timedelta(seconds=6000),
    datetime.now(tz=UTC),
    Processor(ProcessorName.MEAN, 20),
)
plt.plot(df_mean["date"], df_mean["val"])
plt.xlabel("time")
plt.ylabel("val (mean, 20s bins)")
plt.tight_layout()
# NBVAL_IGNORE_OUTPUT

## Raw PB Response

For full control you can fetch the raw protobuf response with `get_data_raw` and decode it yourself with `parse_pb_data`. This is the building block used by `get_events` and `get_data`.

In [ ]:
from epicsarchiver.retrieval.pb import parse_pb_data

# get_data_raw returns the underlying HTTP response; its body is the raw
# Archiver Appliance PB byte stream, which parse_pb_data turns into events.
response = archiver.get_data_raw(
    pv, datetime.now(tz=UTC) - timedelta(seconds=30), datetime.now(tz=UTC)
)
meta, raw_events = parse_pb_data(response.content)
meta

## Searching for PV Names

`search` returns the PV names matching a regex pattern. Optionally restrict the results to PVs that recorded data in a time range with `start`/`end`, and cap the number of results with `limit`.

In [ ]:
# Optionally pass start/end to filter by time range and limit to cap results.
archiver.search("EXAMPLE:.*", limit=10)

## Exporting Events to Other Formats

`write_events` serialises a list of events to a binary stream in one of the `Format` options used by the `export` command: `JSON`, `CSV`, `ARROW`, or `PARQUET`.

In [ ]:
import io

from epicsarchiver.write.export_format import Format, write_events

# Write the events to an in-memory buffer as CSV. Format also supports
# JSON, ARROW, and PARQUET. These formats require the [polars] extra.
buffer = io.BytesIO()
write_events(buffer, Format.CSV, events=raw_events, meta=meta)
print("\n".join(buffer.getvalue().decode().splitlines()[:4]))

## Reading a Local PB File

`read_pb_file` parses an Archiver Appliance `.pb` file from disk into the same `(metadata, events)` tuple returned by `parse_pb_data`, without contacting a server.

In [ ]:
from pathlib import Path

from epicsarchiver.retrieval.pb import read_pb_file

# Persist the raw response to a local .pb file, then read it back.
_pb_path = Path("example.pb")
_pb_path.write_bytes(response.content)
file_meta, file_events = read_pb_file(str(_pb_path))
_pb_path.unlink()  # clean up the temporary file
file_events[:3]

In [ ]:
_mock_server.shutdown()
